In [1]:
import glob
import pandas as pd

In [2]:
reviews_pattern = "../data/raw/reviews_*.csv"
products_pattern = "../data/raw/product_info.csv"

products_df = pd.read_csv(products_pattern)

files = glob.glob(reviews_pattern)
dfs = []
for file in files:
    df = pd.read_csv(file, engine="python")
    dfs.append(df)

df = pd.concat(dfs, ignore_index=True)
df = df.merge(products_df, on="product_id", how="left")

In [3]:
pd.set_option("display.max_columns", None)
df.head(2)

,Unnamed: 0,author_id,LABEL-simple_rating,is_recommended,helpfulness,total_feedback_count,total_neg_feedback_count,total_pos_feedback_count,submission_time,review_text,review_title,skin_tone,eye_color,skin_type,hair_color,product_id,product_name_x,brand_name_x,price_usd_x,product_name_y,brand_id,brand_name_y,loves_count,rating,reviews,size,variation_type,variation_value,variation_desc,ingredients,price_usd_y,value_price_usd,sale_price_usd,limited_edition,new,online_only,out_of_stock,sephora_exclusive,highlights,primary_category,secondary_category,tertiary_category,child_count,child_max_price,child_min_price
0,0,1741593524,5,1.0,1.0,2.0,0,2,2023-02-01,I use this with the Nudestix “Citrus Clean Bal...,Taught me how to double cleanse!,NaN,brown,dry,black,P504322,Gentle Hydra-Gel Face Cleanser,NUDESTIX,19.0,Gentle Hydra-Gel Face Cleanser,7055.0,NUDESTIX,177.0,5.0000,1.0,2.4 oz / 70 ml,Size,2.4 oz / 70 ml,NaN,"['Water (Aqua), Dipropylene Glycol, Peg-6 Capr...",19.0,NaN,NaN,0.0,0.0,1.0,0.0,0.0,['Clean at Sephora'],Skincare,Cleansers,NaN,0.0,NaN,NaN
1,1,31423088263,1-2,0.0,NaN,0.0,0,0,2023-03-21,I bought this lip mask after reading the revie...,Disappointed,NaN,NaN,NaN,NaN,P420652,Lip Sleeping Mask Intense Hydration with Vitam...,LANEIGE,24.0,Lip Sleeping Mask Intense Hydration with Vitam...,6125.0,LANEIGE,1081315.0,4.3508,16118.0,0.7 oz/ 20 g,Color,Original,NaN,"['Diisostearyl Malate, Hydrogenated Polyisobut...",24.0,NaN,NaN,0.0,0.0,0.0,0.0,1.0,"['allure 2019 Best of Beauty Award Winner', 'C...",Skincare,Lip Balms & Treatments,NaN,3.0,24.0,24.0


In [4]:
target = "LABEL-simple_rating"
ids = ["author_id", "product_id", "Unnamed: 0"]

df = df.drop(columns=ids)

### Ile znajduje się w zbiorze cech kategorycznych, a ile numerycznych? 


### Product Info Schema:

- product_id - identyfikator produktu, zmienna tekstowa
- product_name - nazwa produktu, zmienna tekstowa
- brand_id - identyfikator marki, zmienna tekstowa
- brand_name - nazwa marki, zmienna tekstowa
- loves_count - liczba polubień, zmienna numeryczna
- rating - ocena, zmienna numeryczna
- reviews - liczba opinii, zmienna numeryczna
- size - rozmiar, zmienna tekstowa
- variation_type - typ wariacji, zmienna tekstowa
- variation_value - wartość wariacji, zmienna tekstowa
- variation_desc - opis wariacji, zmienna tekstowa
- ingredients - skład, zmienna tekstowa
- price_usd - cena w USD, zmienna numeryczna
- value_price_usd - wartość ceny w USD, zmienna numeryczna
- sale_price_usd - cena promocyjna w USD, zmienna numeryczna
- limited_edition - edycja limitowana, zmienna logiczna
- new - nowy, zmienna logiczna
- online_only - dostępny tylko online, zmienna logiczna
- out_of_stock - brak w magazynie, zmienna logiczna
- sephora_exclusive - ekskluzywny dla Sephory, zmienna logiczna
- highlights - wyróżnienia, zmienna tekstowa
- primary_category - kategoria główna, zmienna tekstowa
- secondary_category - kategoria вторsza, zmienna tekstowa
- tertiary_category - kategoria trzeciorzędowa, zmienna tekstowa
- child_count - liczba produktów potomnych, zmienna numeryczna
- child_max_price - maksymalna cena produktów potomnych, zmienna numeryczna
- child_min_price - minimalna cena produktów potomnych, zmienna numeryczna

### Review Info Schema:
- Unnamed: 0 - review_id
- author_id - identyfikator autora, zmienna tekstowa
- LABEL-simple_rating - target, jako zmienna kategoryczna (negatywna, neutralna, pozytywna)
- is_recommended - czy polecane, zmienna logiczna
- helpfulness - pomocność, zmienna numeryczna
- total_feedback_count - całkowita liczba opinii, zmienna numeryczna
- total_neg_feedback_count - liczba negatywnych opinii, zmienna numeryczna
- total_pos_feedback_count - liczba pozytywnych opinii, zmienna numeryczna
- submission_time - czas przesłania, zmienna tekstowa
- review_text - tekst opinii, zmienna tekstowa
- review_title - tytuł opinii, zmienna tekstowa
- skin_tone - odcień skóry, zmienna tekstowa
- eye_color - kolor oczu, zmienna tekstowa
- skin_type - typ skóry, zmienna tekstowa
- hair_color - kolor włosów, zmienna tekstowa
- product_id - identyfikator produktu, zmienna tekstowa
- product_name - nazwa produktu, zmienna tekstowa
- brand_name - nazwa marki, zmienna tekstowa
- price_usd - cena w USD, zmienna numeryczna

### Czy zmienna wyjściowa jest kategoryczna, czy numeryczna? 

In [9]:
target = "LABEL-simple_rating"
df[target].value_counts()

LABEL-simple_rating
5      698951
3-4    281205
1-2    114255
Name: count, dtype: int64

zmienna wyjściowa, czyli target, przyjmuje 3 wartości:

- 1-2 w liczbie 1142255
- 3-4 w liczbie 281205
- 5 w liczbie 604540

traktowana jako zmienna kategoryczna, ponieważ reprezentuje trzy kategorie ocen: negatywna (1-2), neutralna (3-4) i pozytywna (5).


### Czy i ile w zbiorze jest brakujących wartości? Dla jakich zmiennych? Co z tego wynika? Jakie są możliwe sposoby radzenia sobie z brakującymi wartościami? 

In [13]:
result = pd.concat([df.isna().sum(), df.isna().mean() * 100], axis=1)
result.columns = ["missing_count", "missing_percent"]
result

,missing_count,missing_percent
LABEL-simple_rating,701,0.064012
is_recommended,168689,15.403813
helpfulness,562041,51.322696
total_feedback_count,701,0.064012
total_neg_feedback_count,275,0.025112
total_pos_feedback_count,275,0.025112
submission_time,275,0.025112
review_text,1719,0.156970
review_title,311750,28.467408
skin_tone,171240,15.636757


procent braków znacząco rózni się w zaleznosci od kolumny. Wiekoszosc kolumn jednak ma procent braków bliski 0. W przypadku braków w kolumnach tekstowych, można rozważyć uzupełnienie ich wartością np. "Unknown". W przypadku kolumn numerycznych, można rozważyć uzupełnienie braków średnią lub medianą, lub też usunięcie wierszy z brakującymi wartościami, jeśli ich liczba jest niewielka.

### Podejście do wstępnego przetwarzania danych:

Usunięcie kolumn z:
- ID
- wysokim procentem braków

Uzupełnienie braków w kolumnach tekstowych wartością "Unknown". 